In [1]:
import numpy as np
np.random.seed(42)

In [ ]:
# 15 samples, 5 input features
X = np.array([
    [1.0, 1.2, 0.8, 1.1, 0.9],
    [1.2, 0.9, 1.1, 1.0, 1.3],
    [0.8, 1.1, 1.0, 0.9, 1.2],
    [1.1, 1.0, 0.9, 1.2, 1.0],
    [0.9, 1.3, 1.2, 1.0, 1.1],

    [4.0, 4.2, 3.8, 4.1, 4.3],
    [4.2, 3.9, 4.1, 4.0, 3.8],
    [3.8, 4.1, 4.0, 4.3, 4.2],
    [4.1, 4.0, 4.2, 3.9, 4.1],
    [3.9, 4.3, 4.1, 4.2, 4.0],

    [7.0, 7.2, 6.8, 7.1, 7.3],
    [7.2, 6.9, 7.1, 7.0, 6.8],
    [6.8, 7.1, 7.0, 7.3, 7.2],
    [7.1, 7.0, 7.2, 6.9, 7.1],
    [6.9, 7.3, 7.1, 7.2, 7.0]
])

# One-hot encoded target values
Y = np.array([
    [1, 0, 0], [1, 0, 0], [1, 0, 0], [1, 0, 0], [1, 0, 0],
    [0, 1, 0], [0, 1, 0], [0, 1, 0], [0, 1, 0], [0, 1, 0],
    [0, 0, 1], [0, 0, 1], [0, 0, 1], [0, 0, 1], [0, 0, 1]
])

print('Input shape:', X.shape)
print('Target shape:', Y.shape)

In [ ]:
def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return (x > 0).astype(float)


def softmax(x):
    # Subtracting the maximum keeps the calculation numerically stable.
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [ ]:
# Network architecture: 5 -> 5 -> 5 -> 3

W1 = np.random.randn(5, 5) * 0.1
b1 = np.zeros((1, 5))

W2 = np.random.randn(5, 5) * 0.1
b2 = np.zeros((1, 5))

W3 = np.random.randn(5, 3) * 0.1
b3 = np.zeros((1, 3))

print('W1:', W1.shape)
print('W2:', W2.shape)
print('W3:', W3.shape)

In [ ]:
def forward(X):
    # Layer 1
    Z1 = X @ W1 + b1
    A1 = relu(Z1)

    # Layer 2
    Z2 = A1 @ W2 + b2
    A2 = relu(Z2)

    # Output layer
    Z3 = A2 @ W3 + b3
    A3 = softmax(Z3)

    # Save intermediate values for backpropagation
    cache = (Z1, A1, Z2, A2, Z3, A3)
    return A3, cache


probabilities, cache = forward(X)

print('Output probabilities for first 5 samples:')
print(probabilities[:5])
print('\nSum of probabilities for first sample:', np.sum(probabilities[0]))

In [ ]:
def cross_entropy_loss(Y, predictions):
    epsilon = 1e-12
    predictions = np.clip(predictions, epsilon, 1 - epsilon)
    return -np.mean(np.sum(Y * np.log(predictions), axis=1))


loss = cross_entropy_loss(Y, probabilities)
print('Initial loss:', loss)

In [ ]:
def backward(X, Y, cache):
    Z1, A1, Z2, A2, Z3, A3 = cache
    m = X.shape[0]

    # Output layer gradient
    dZ3 = (A3 - Y) / m
    dW3 = A2.T @ dZ3
    db3 = np.sum(dZ3, axis=0, keepdims=True)

    # Hidden layer 2 gradient
    dA2 = dZ3 @ W3.T
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    # Hidden layer 1 gradient
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    gradients = (dW1, db1, dW2, db2, dW3, db3)
    return gradients

In [ ]:
learning_rate = 0.1
epochs = 5000

loss_history = []

for epoch in range(epochs):
    # Forward pass
    predictions, cache = forward(X)

    # Calculate loss
    loss = cross_entropy_loss(Y, predictions)
    loss_history.append(loss)

    # Backward pass
    dW1, db1, dW2, db2, dW3, db3 = backward(X, Y, cache)

    # Gradient descent update
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3

    if epoch % 500 == 0:
        print(f'Epoch {epoch}, Loss = {loss:.4f}')

In [ ]:
# Simple plot using matplotlib only for visualization
import matplotlib.pyplot as plt

plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
predictions, _ = forward(X)

# Convert one-hot targets and probabilities to class numbers
actual_classes = np.argmax(Y, axis=1)
predicted_classes = np.argmax(predictions, axis=1)

print('Sample predictions:')
for i in range(len(X)):
    print(
        f'Sample {i+1}: '
        f'Probabilities = {np.round(predictions[i], 3)}, '
        f'Predicted = Class {predicted_classes[i]}, '
        f'Actual = Class {actual_classes[i]}'
    )

accuracy = np.mean(predicted_classes == actual_classes)
print(f'\nTraining accuracy: {accuracy * 100:.2f}%')

In [ ]:
new_input = np.array([[4.1, 4.0, 4.2, 4.1, 4.0]])

new_probabilities, _ = forward(new_input)
predicted_class = np.argmax(new_probabilities[0])

print('Input:', new_input)
print('Class probabilities:', np.round(new_probabilities[0], 4))
print('Predicted class:', predicted_class)